# Simulación del estómago — Fase 4: partículas de comida (arrastre, rotura y erosión)

**Qué añade esta fase.** Es el acoplamiento químico–mecánico que planteaste al principio. La comida deja de ser un líquido o un bolo inmóvil y pasa a ser una **población de partículas** que:
1. **Se mueven** con el flujo de la fase 3 (arrastre) y sedimentan.
2. **Se rompen** cuando una contracción antral las aprieta con más fuerza de la que resisten. El ácido las ablanda, así que la química acelera la mecánica.
3. **Se erosionan** por la superficie: la pepsina disuelve la capa exterior, con una velocidad que depende del pH.
4. **Salen por el píloro** solo si son menores de 1–2 mm. Es la **criba pilórica**.

**Dos descripciones de lo mismo, para verificarse mutuamente.**
- **Lagrangiana** (Monte Carlo): cada partícula numérica tiene un tamaño y sufre eventos aleatorios de rotura, erosión y salida.
- **Balance de poblaciones** (PBM): se sigue el *número* de partículas en cada intervalo de tamaño mediante un sistema de EDOs.

Si las dos dan lo mismo con las mismas leyes, ambas implementaciones son correctas.

**Por qué importa para la validación.** En la fase 1b, el error de modelo dominante era el vaciado demasiado rápido de la comida de consenso: retención a 2 h del 2 % frente al 30–60 % medido. El diagnóstico era que el modelo vaciaba la comida como un líquido. Aquí probamos si **la criba de partículas**, que retiene lo grande hasta triturarlo, corrige ese error.

**Uso en Colab**: `Ejecutar todo`, unos 3–4 minutos. Si subes el `campo_antro.npz` de la fase 3, el cuaderno lo usa. Si no, lo regenera (≈ 75 s).

## 1. Ecuaciones

### 1.1 Movimiento de una partícula (arrastre y sedimentación)
Para una esfera de diámetro $d$ y densidad $\rho_p$ en un fluido de densidad $\rho_f$ y viscosidad $\mu = \rho_f \nu$ usamos la forma simplificada de la ecuación de Maxey–Riley:
$$ \frac{d\mathbf{v}}{dt} = \frac{\mathbf{u}(\mathbf{x}_p, t) - \mathbf{v}}{\tau_p} + \Big(1 - \frac{\rho_f}{\rho_p}\Big)\mathbf{g}, \qquad \tau_p = \frac{\rho_p d^2}{18\,\mu\, f(\mathrm{Re}_p)}, \qquad f = 1 + 0.15\,\mathrm{Re}_p^{0.687}. $$
$f$ es la corrección de Schiller–Naumann, para cuando el número de Reynolds de la partícula, $\mathrm{Re}_p = |\mathbf{u}-\mathbf{v}| d/\nu$, no es pequeño.

Con $\mathbf{u}$ congelada durante el paso, la ecuación es lineal y su solución exacta es
$$ \mathbf{v}(t+\Delta t) = \mathbf{u} + \mathbf{v}_t + (\mathbf{v} - \mathbf{u} - \mathbf{v}_t)\,e^{-\Delta t/\tau_p}, \qquad \mathbf{v}_t = \tau_p\Big(1 - \frac{\rho_f}{\rho_p}\Big)\mathbf{g}. $$
La usamos como integrador: es estable aunque $\tau_p \ll \Delta t$, que es lo habitual para partículas de milímetros en un fluido viscoso.

**Hipótesis.** Acoplamiento en un solo sentido: las partículas no alteran el flujo, lo que equivale a suspensión diluida. Se desprecian la masa añadida, la fuerza de Basset y la sustentación. La gravedad actúa hacia $-y$.

### 1.2 Rotura por contacto con la pared
Las tensiones del fluido son minúsculas frente a la resistencia de la comida: lo comprobaremos en la sección 4. Lo que rompe las partículas es el **contacto** con la pared cuando una contracción antral las atrapa contra la oclusión terminal. La frecuencia de rotura (función de selección) de una partícula de tamaño $d$ es
$$ S(d, t) = \underbrace{f_{TA}(d)}_{\text{visitas a la zona de molienda (LBM)}} \times \underbrace{q}_{\text{queda atrapada}} \times \underbrace{H(d - g_{min})}_{\text{no cabe por el hueco}} \times \underbrace{P_{rot}(d, t)}_{\text{la fuerza supera su resistencia}} . $$
Para la última probabilidad, la fuerza de contracción $F$ se reparte uniformemente entre 0.2 y 2 N (fuerzas medidas en el estómago humano), y la partícula rompe si $F/d^2 > \sigma_c$:
$$ P_{rot} = \mathrm{clip}\!\left(\frac{F_{max} - \sigma_c d^2}{F_{max} - F_{min}},\,0,\,1\right). $$
La rotura es **binaria en mitades iguales de volumen**: $d \to d/2^{1/3}$, y cada partícula da dos.

### 1.3 Acoplamiento químico: el ácido debilita la comida
La resistencia cae con la exposición acumulada al ácido:
$$ \sigma_c(t) = \sigma_{c0}\big[w_\infty + (1 - w_\infty)\,e^{-X(t)/X_w}\big], \qquad X(t) = \int_0^t \max(0,\ \mathrm{pH}_{ref} - \mathrm{pH})\,dt' . $$
El pH sigue la curva de la fase 1b para la comida de consenso: baja de 5.8 a ~1 con su punto medio hacia los 30 min.

### 1.4 Erosión superficial
La pepsina disuelve la superficie, que retrocede a una velocidad uniforme que depende del pH:
$$ \frac{dd}{dt} = -r_e(\mathrm{pH}) = -r_{e0}\, \exp\!\Big[-\Big(\frac{\mathrm{pH} - \mathrm{pH}_{opt}}{\sigma}\Big)^2\Big]. $$
Es el modelo de **núcleo que se encoge** (*shrinking core*). La masa erosionada pasa a disolución y sale con el líquido.

### 1.5 Criba pilórica
Solo las partículas con $d < d_c$, entre 1 y 2 mm, salen con el líquido, al ritmo de vaciado de la fase 1b: $k(t) = Q(t)/V(t)$. Las mayores vuelven atrás con el chorro retropulsivo.

### 1.6 Balance de poblaciones discreto
Usamos una malla de tamaños **geométrica en volumen**, con $k$ celdas por cada mitad de volumen: $v_{i+k} = 2\,v_i$. Con ella, la rotura binaria lleva **exactamente** de la celda $i$ a la $i - k$, sin error de discretización. Para el número de partículas $N_i$:
$$ \frac{dN_i}{dt} = \underbrace{-S_i N_i + 2\,S_{i+k} N_{i+k}}_{\text{rotura}} \;\underbrace{-\,e_i N_i + e_{i+1} N_{i+1}}_{\text{erosión (upwind en tamaño)}} \;\underbrace{-\,k(t)\,N_i\,\mathbb{1}[d_i < d_c]}_{\text{criba}}, \qquad e_i = \frac{r_e}{d_i - d_{i-1}} . $$
La masa que la erosión saca de cada celda pasa a la reserva disuelta. Con eso, la **masa total se conserva exactamente**: sólida + disuelta + vaciada.

### 1.7 Métrica comparable con la gammagrafía
El marcador radiactivo de la comida de consenso va unido a la clara de huevo. La retención medida es, por tanto, **sólido en el estómago + material disuelto que aún no ha salido**.

## 2. Código

In [ ]:
import math, time, os
from dataclasses import dataclass, replace

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from scipy.stats import qmc
from IPython.display import display

plt.rcParams.update({
    "figure.dpi": 110, "figure.figsize": (11, 4.2),
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.25, "grid.linewidth": 0.6,
    "lines.linewidth": 2.0, "font.size": 10.5, "legend.frameon": False,
})
GRIS = "#3b3b3b"; GRIS_MEDIO = "#9a9a96"; TINTA2 = "#52514e"
C1, C2, C3, C4, C5 = "#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#6250d6"
pd.set_option("display.float_format", lambda x: f"{x:.4g}")
VERIFICACION = []

### 2.1 Balance de poblaciones

In [ ]:
@dataclass
class ParamPart:
    # --- Comida y partículas (tamaños en mm, tiempo en min) ---
    rho_p: float = 1.05e-3        # g/mm³   densidad de la partícula (1.05 g/cm³)
    d50_ini: float = 3.0          # mm      mediana tras la masticación ("hasta 3 mm")
    sg_ini: float = 0.5           # —       anchura lognormal (ln) de la distribución inicial
    # --- Trituración por contacto con la pared (antro terminal) ---
    f_TA: object = 0.5            # 1/min   visitas a la zona de molienda: número o tabla (d, f) de la sección 4
    q_atrapada: float = 0.5       # —       probabilidad de quedar atrapada en la oclusión terminal en cada visita
    g_min: float = 0.5            # mm      hueco mínimo de la oclusión: más pequeñas escapan
    F_min: float = 0.2            # N       rango de fuerzas de contracción (Kamba et al.)
    F_max: float = 2.0            # N
    sigma_c0: float = 1.0e5       # Pa      resistencia a la fractura sin digerir (supuesto)
    w_inf: float = 0.3            # —       resistencia residual tras exposición ácida larga
    X_w: float = 60.0             # pH·min  exposición ácida que debilita por un factor e
    pH_ref: float = 4.0           # —       por encima de este pH no hay debilitamiento
    # --- Erosión química superficial ---
    r_e0: float = 5e-3            # mm/min  velocidad de retroceso de la superficie a pH óptimo
    pH_opt: float = 2.0
    sigma_pH: float = 1.5
    # --- Criba pilórica y vaciado del líquido ---
    d_c: float = 1.5              # mm      tamaño de corte del píloro (< 1–2 mm pasan)
    kcal_min: float = 2.0         # kcal/min ritmo energético de vaciado del líquido (fase 1b)
    V_res: float = 30.0           # mL
    q_basal: float = 0.25         # mL/min  caudal basal tras vaciar la comida
    # --- Curva de pH (aproximación de la fase 1b para la comida de consenso) ---
    pH_0: float = 5.8
    pH_fin: float = 1.0
    t_pH: float = 30.0            # min     instante en que el pH pasa por el punto medio
    s_pH: float = 6.0             # min     anchura de la caída


def pH_t(t, pp):
    return pp.pH_fin + (pp.pH_0 - pp.pH_fin) / (1 + np.exp((t - pp.t_pH) / pp.s_pH))


def tasa_erosion(t, pp):
    pH = pH_t(t, pp)
    return pp.r_e0 * np.exp(-((pH - pp.pH_opt) / pp.sigma_pH) ** 2)          # mm/min


def resistencia(X, pp):
    return pp.sigma_c0 * (pp.w_inf + (1 - pp.w_inf) * np.exp(-X / pp.X_w))   # Pa


def prob_rotura(d_mm, sigma_c, pp):
    # Fuerza de contracción uniforme en [F_min, F_max]; rompe si F/d² > sigma_c  ->  F > sigma_c d²
    F_nec = sigma_c * (d_mm * 1e-3) ** 2
    return np.clip((pp.F_max - F_nec) / (pp.F_max - pp.F_min), 0.0, 1.0)


def frecuencia_TA(d_mm, pp):
    # f_TA puede ser un número o una tabla (d_mm, f) medida con las partículas en el flujo LBM
    if np.isscalar(pp.f_TA):
        return pp.f_TA + 0.0 * d_mm
    d_tab, f_tab = pp.f_TA
    return np.interp(d_mm, d_tab, f_tab)


def k_vaciado(t, V0, T_e, pp):
    # tasa de salida del contenido líquido (1/min): volumen que baja linealmente hasta V_res en T_e
    if t < T_e:
        V = V0 - (V0 - pp.V_res) * t / T_e
        return (V0 - pp.V_res) / T_e / V
    return pp.q_basal / pp.V_res


class MallaTamanos:
    # Malla geométrica en volumen: k celdas por cada mitad de volumen (d_i = d_min 2^{i/(3k)}).
    # Con esta malla, la rotura binaria en mitades iguales lleva EXACTAMENTE de la celda i a la i-k.
    def __init__(self, d_min=0.05, d_max=10.0, k=1):
        n = int(math.ceil(3 * k * math.log2(d_max / d_min))) + 1
        self.k = k
        self.d = d_min * 2 ** (np.arange(n) / (3 * k))
        self.n = n


def distribucion_inicial(malla, masa_g, pp):
    # Lognormal en masa: fracción de masa en cada celda ∝ densidad lognormal de ln d
    ld = np.log(malla.d)
    w = np.exp(-0.5 * ((ld - np.log(pp.d50_ini)) / pp.sg_ini) ** 2)
    w = w / w.sum()
    m = pp.rho_p * np.pi * malla.d ** 3 / 6
    return masa_g * w / m                                                    # número de partículas


def rhs_pbm(t, y, malla, pp, V0, T_e, opciones):
    # Estado: [N_0..N_{n-1}, M_disuelta, X_exposición, sale_sólido, sale_disuelto]
    n, k = malla.n, malla.k
    N = y[:n]; Mdis = y[n]; X = y[n + 1]
    d = malla.d; m = pp.rho_p * np.pi * d ** 3 / 6
    dN = np.zeros(n)
    # --- rotura binaria ---
    if opciones.get("rotura", True):
        sig = resistencia(X, pp) if opciones.get("debilitamiento", True) else pp.sigma_c0
        S = opciones.get("S_cte")
        if S is None:
            S = frecuencia_TA(d, pp) * pp.q_atrapada * (d > pp.g_min) * prob_rotura(d, sig, pp)
        else:
            S = np.full(n, S); S[:k] = 0.0                                   # la celda más pequeña no se rompe
        perdida = S * N
        dN -= perdida
        dN[:-k] += 2 * perdida[k:]
    # --- erosión superficial (advección en tamaño, upwind) ---
    gan_dis = 0.0
    if opciones.get("erosion", True):
        r = opciones.get("r_cte", None)
        r = tasa_erosion(t, pp) if r is None else r
        dd = np.diff(np.concatenate([[0.0], d]))                             # d_i - d_{i-1} (d_{-1} = 0)
        e = r / dd                                                           # 1/min
        flujo = e * N
        dN -= flujo
        dN[:-1] += flujo[1:]
        m_prev = np.concatenate([[0.0], m[:-1]])
        gan_dis = np.sum(flujo * (m - m_prev))                               # masa que pasa a disolución
    # --- criba pilórica + vaciado ---
    sale_s = sale_d = 0.0
    if opciones.get("vaciado", True):
        kv = k_vaciado(t, V0, T_e, pp)
        pasa = d < pp.d_c
        dN -= kv * N * pasa
        sale_s = kv * np.sum(N * m * pasa)
        sale_d = kv * Mdis
    pH = pH_t(t, pp)
    dX = max(0.0, pp.pH_ref - pH)
    return np.concatenate([dN, [gan_dis - sale_d, dX, sale_s, sale_d]])


def resolver_pbm(malla, pp, masa_g, V0, T_e, t_final, opciones=None, N0=None, t_eval=None, rtol=1e-8):
    opciones = opciones or {}
    N0 = distribucion_inicial(malla, masa_g, pp) if N0 is None else N0
    y0 = np.concatenate([N0, [0.0, 0.0, 0.0, 0.0]])
    # el vaciado cambia de régimen en T_e: se integra en dos tramos para no saltar la discontinuidad
    cortes = [0.0] + ([T_e] if 0 < T_e < t_final else []) + [t_final]
    ts, ys = [], []
    t_eval = np.linspace(0, t_final, 241) if t_eval is None else t_eval
    for a, b in zip(cortes[:-1], cortes[1:]):
        te = t_eval[(t_eval >= a) & (t_eval <= b)]
        sol = solve_ivp(rhs_pbm, (a, b), y0, method="LSODA", rtol=rtol, atol=1e-10 * max(1.0, N0.max()),
                        args=(malla, pp, V0, T_e, opciones), dense_output=True)
        assert sol.success, sol.message
        ts.append(te); ys.append(sol.sol(te).T); y0 = sol.y[:, -1]
    t = np.concatenate(ts); Y = np.vstack(ys)
    t, i = np.unique(t, return_index=True)
    return t, Y[i]


def masas_pbm(Y, malla, pp):
    n = malla.n
    m = pp.rho_p * np.pi * malla.d ** 3 / 6
    solido = Y[:, :n] @ m
    return {"sólido": solido, "disuelto": Y[:, n], "salida sólida": Y[:, n + 2], "salida disuelta": Y[:, n + 3]}


def d50_masa(Nrow, malla, pp):
    m = pp.rho_p * np.pi * malla.d ** 3 / 6
    w = Nrow * m
    if w.sum() <= 0:
        return np.nan
    c = np.cumsum(w) / w.sum()
    return float(np.interp(0.5, c, malla.d))

### 2.2 Monte Carlo lagrangiano (mismas leyes)

In [ ]:
def mc_lagrangiano(pp, masa_g, V0, T_e, t_final, dt=0.1, n_part=20000, opciones=None, semilla=0,
                   n_max=200000, d_ini=None, guardar_cada=1.0):
    # Monte Carlo lagrangiano de las mismas leyes que el balance de poblaciones:
    # cada partícula numérica representa `w` partículas reales y sufre eventos aleatorios.
    opciones = opciones or {}
    rng = np.random.default_rng(semilla)
    if d_ini is None:
        # la distribución inicial es lognormal EN MASA con mediana d50: en número, la mediana es d50·exp(-3 sg²)
        d = np.exp(rng.normal(np.log(pp.d50_ini) - 3 * pp.sg_ini ** 2, pp.sg_ini, n_part))
    else:
        d = np.full(n_part, float(d_ini))
    masa = lambda dd: pp.rho_p * np.pi * dd ** 3 / 6
    w = np.full(d.size, masa_g / masa(d).sum())                           # peso de cada partícula numérica
    Mdis = X = sale_s = sale_d = 0.0
    n_pasos = int(round(t_final / dt))
    cada = max(1, int(round(guardar_cada / dt)))
    reg = {"t": [], "sólido": [], "disuelto": [], "salida sólida": [], "salida disuelta": [], "n": [], "d50": []}
    reg["d_final"] = None
    for paso in range(n_pasos + 1):
        t = paso * dt
        if paso % cada == 0:
            ms = masa(d)
            reg["t"].append(t); reg["sólido"].append(float(np.sum(w * ms))); reg["disuelto"].append(Mdis)
            reg["salida sólida"].append(sale_s); reg["salida disuelta"].append(sale_d); reg["n"].append(d.size)
            if d.size:
                o = np.argsort(d); c = np.cumsum((w * ms)[o]) / np.sum(w * ms)
                reg["d50"].append(float(np.interp(0.5, c, d[o])))
            else:
                reg["d50"].append(np.nan)
        if paso == n_pasos:
            break
        tm = t + 0.5 * dt                                                    # tasas evaluadas a mitad de paso
        # rotura
        if opciones.get("rotura", True) and d.size:
            sig = resistencia(X, pp) if opciones.get("debilitamiento", True) else pp.sigma_c0
            S = opciones.get("S_cte")
            if S is None:
                S = frecuencia_TA(d, pp) * pp.q_atrapada * (d > pp.g_min) * prob_rotura(d, sig, pp)
            else:
                S = np.where(d > opciones.get("d_min_rotura", 0.0), S, 0.0)
            rompe = rng.random(d.size) < 1 - np.exp(-S * dt)
            d = np.concatenate([np.where(rompe, d * 2 ** (-1 / 3), d), d[rompe] * 2 ** (-1 / 3)])
            w = np.concatenate([w, w[rompe]])
        # erosión
        if opciones.get("erosion", True) and d.size:
            r = opciones.get("r_cte", None)
            r = tasa_erosion(tm, pp) if r is None else r
            d_nuevo = d - r * dt
            Mdis += float(np.sum(w * (masa(d) - masa(np.clip(d_nuevo, 0, None)))))
            vivo = d_nuevo > 0
            d, w = d_nuevo[vivo], w[vivo]
        # criba + vaciado
        if opciones.get("vaciado", True):
            kv = k_vaciado(tm, V0, T_e, pp)
            p_sal = 1 - np.exp(-kv * dt)
            sale = (d < pp.d_c) & (rng.random(d.size) < p_sal)
            sale_s += float(np.sum(w[sale] * masa(d[sale])))
            d, w = d[~sale], w[~sale]
            sale_d += Mdis * p_sal; Mdis *= (1 - p_sal)
        X += max(0.0, pp.pH_ref - float(pH_t(tm, pp))) * dt
        # control de tamaño: si hay demasiadas partículas numéricas, se descarta al azar la mitad
        # SOLO en las clases de tamaño muy pobladas (y se duplica su peso). Así no se pierden al azar
        # las pocas partículas grandes que llevan mucha masa (eso daría mucho ruido en la masa).
        if d.size > n_max:
            clase = np.floor(3 * np.log2(d)).astype(int)
            _, inv, cuenta = np.unique(clase, return_inverse=True, return_counts=True)
            poblada = cuenta[inv] > 200
            quita = poblada & (rng.random(d.size) < 0.5)
            w = np.where(poblada, 2 * w, w)
            d, w = d[~quita], w[~quita]
    out = {k: np.array(v) for k, v in reg.items() if k != "d_final"}
    out["d_final"], out["w"] = d, w
    return out

### 2.3 Partículas en el flujo de la fase 3

In [ ]:
def a_numpy(a):
    # Convierte un array de cualquier backend (NumPy, CuPy, JAX) a NumPy
    if hasattr(a, "get"):
        return a.get()              # CuPy
    return np.asarray(a)            # NumPy y JAX


def interpolar_bilineal(campo, xs, ys):
    # campo (ny, nx) en nodos de red; xs, ys en coordenadas de red (centro del nodo i en x = i)
    ny, nx = campo.shape
    x = np.clip(xs, 0, nx - 1.001); y = np.clip(ys, 0, ny - 1.001)
    i0 = np.floor(x).astype(int); j0 = np.floor(y).astype(int)
    tx = x - i0; ty = y - j0
    c00 = campo[j0, i0]; c10 = campo[j0, i0 + 1]; c01 = campo[j0 + 1, i0]; c11 = campo[j0 + 1, i0 + 1]
    return (1 - tx) * (1 - ty) * c00 + tx * (1 - ty) * c10 + (1 - tx) * ty * c01 + tx * ty * c11

In [ ]:
class CampoPeriodico:
    # Campo de velocidades de un periodo de onda (fase 3), repetido periódicamente en el tiempo.
    def __init__(self, datos):
        self.t = datos["t"]; self.ux = datos["ux"]; self.uy = datos["uy"]; self.sol = datos["solido"]
        self.x = datos["x"]; self.y = datos["y"]; self.dx = float(datos["dx"]); self.T = float(datos["periodo"])
        self.nu = float(datos["nu"])
        self.tau = datos["tau_max"] if "tau_max" in datos else None
        self.nf = len(self.t) - 1                    # el último marco repite el primero (t = T)
        self.dtf = self.t[1] - self.t[0]
        # hueco local (cm) en cada columna y marco: nº de nodos fluidos × dx
        self.hueco = (~self.sol).sum(axis=1) * self.dx

    def _marco(self, t):
        s = (t % self.T) / self.dtf
        k = int(np.floor(s)) % self.nf
        return k, s - np.floor(s)

    def velocidad(self, t, px, py):
        k, a = self._marco(t)
        u0 = self._interp(self.ux[k], px, py); u1 = self._interp(self.ux[k + 1], px, py)
        v0 = self._interp(self.uy[k], px, py); v1 = self._interp(self.uy[k + 1], px, py)
        return (1 - a) * u0 + a * u1, (1 - a) * v0 + a * v1

    def _indices(self, px, py):
        fx = (px - self.x[0]) / self.dx; fy = (py - self.y[0]) / self.dx
        return fx, fy

    def _interp(self, campo, px, py):
        fx, fy = self._indices(px, py)
        return interpolar_bilineal(campo, fx, fy)

    def solido_en(self, t, px, py):
        k, a = self._marco(t)
        sol = self.sol[k if a < 0.5 else k + 1]
        fx, fy = self._indices(px, py)
        i = np.clip(np.rint(fx).astype(int), 0, sol.shape[1] - 1); j = np.clip(np.rint(fy).astype(int), 0, sol.shape[0] - 1)
        return sol[j, i], sol, i, j

    def hueco_en(self, t, px):
        k, _ = self._marco(t)
        fx = (px - self.x[0]) / self.dx
        i = np.clip(np.rint(fx).astype(int), 0, self.hueco.shape[1] - 1)
        return self.hueco[k, i]


def tiempo_relajacion(d_cm, rho_p, rho_f, nu, v_rel):
    # Arrastre de Stokes con corrección de Schiller–Naumann: tau = rho_p d² / (18 mu f(Re_p))
    mu = rho_f * nu
    Re_p = np.abs(v_rel) * d_cm / nu
    f = 1 + 0.15 * Re_p ** 0.687
    return rho_p * d_cm ** 2 / (18 * mu * f)


def seguir_particulas(campo, d_mm, rho_p, rho_f=1.0, g=981.0, t_final=200.0, dt=0.02, semilla=0,
                      x_zona=8.0, frac_cuello=0.6, guardar_cada=0.5):
    # Partículas con inercia y sedimentación (gravedad hacia -y) en el campo periódico.
    # Integración exponencial con el fluido congelado durante dt (exacta para arrastre lineal).
    # Evento de compresión: la partícula está en el antro terminal (x > x_zona) dentro de un cuello
    # con hueco < frac_cuello · H. Se cuenta una vez por paso de onda.
    rng = np.random.default_rng(semilla)
    d_mm = np.asarray(d_mm, float); n = d_mm.size; d_cm = d_mm / 10
    s0 = campo.sol[0]
    cand = np.argwhere(~s0)
    sel = cand[rng.choice(len(cand), n, replace=True)]
    py = campo.y[sel[:, 0]] + rng.uniform(-0.5, 0.5, n) * campo.dx
    px = campo.x[sel[:, 1]] + rng.uniform(-0.5, 0.5, n) * campo.dx
    vx, vy = campo.velocidad(0.0, px, py)
    flot = (1 - rho_f / rho_p) * g
    cuerpo = (campo.x > 3.8) & (campo.x < 4.4)                              # cuerpo gástrico: sin ondas ni dilatación
    H_ref = float(np.median(campo.hueco[:, cuerpo]))
    ultimo_evento = np.full(n, -1e9)
    eventos = np.zeros(n)
    tiempo_zona = np.zeros(n)
    hist_x, hist_y, hist_t = [px.copy()], [py.copy()], [0.0]
    cada = max(1, int(round(guardar_cada / dt)))
    for paso in range(int(round(t_final / dt))):
        t = paso * dt
        ux, uy = campo.velocidad(t, px, py)
        rx, ry = vx - ux, vy - uy
        tau = tiempo_relajacion(d_cm, rho_p, rho_f, campo.nu, np.hypot(rx, ry))
        vtx, vty = 0.0, -tau * flot                                          # velocidad terminal relativa
        e = np.exp(-dt / tau)
        cx_, cy_ = vx - ux - vtx, vy - uy - vty
        nx_ = px + (ux + vtx) * dt + cx_ * tau * (1 - e)
        ny_ = py + (uy + vty) * dt + cy_ * tau * (1 - e)
        vx = ux + vtx + cx_ * e; vy = uy + vty + cy_ * e
        # paredes: si el centro entra en el sólido, se recoloca en el nodo fluido más cercano de su columna
        dentro, sol, i, j = campo.solido_en(t + dt, nx_, ny_)
        if dentro.any():
            for q in np.where(dentro)[0]:
                col = ~sol[:, i[q]]
                if col.any():
                    jj = np.where(col)[0]
                    jn = jj[np.argmin(np.abs(jj - j[q]))]
                    ny_[q] = campo.y[jn]
                else:
                    nx_[q], ny_[q] = px[q], py[q]
                vy[q] = 0.0
        px, py = nx_, ny_
        # eventos de compresión en el antro terminal
        en_zona = px > x_zona
        tiempo_zona += en_zona * dt
        cuello = en_zona & (campo.hueco_en(t + dt, px) < frac_cuello * H_ref)
        nuevo = cuello & (t - ultimo_evento > 0.5 * campo.T)
        eventos += nuevo; ultimo_evento = np.where(nuevo, t, ultimo_evento)
        if (paso + 1) % cada == 0:
            hist_x.append(px.copy()); hist_y.append(py.copy()); hist_t.append(t + dt)
    return {"x": np.array(hist_x), "y": np.array(hist_y), "t": np.array(hist_t), "d_mm": d_mm,
            "eventos_por_min": eventos / (t_final / 60.0), "frac_tiempo_zona": tiempo_zona / t_final}

## 3. Verificación

### T1 — Rotura binaria con tasa constante: solución exacta
Si todas las partículas se rompen a una tasa $S$ constante, partiendo de $N_0$ partículas iguales, el número en la generación $g$ (tras $g$ roturas) es
$$ N_g(t) = N_0\, 2^g\, \frac{(St)^g}{g!}\, e^{-St}, $$
y la **fracción de masa** en la generación $g$ sigue una distribución de Poisson de media $St$. El balance de poblaciones debe reproducirla hasta la tolerancia del integrador. El Monte Carlo debe converger con error $\propto N^{-1/2}$, el ritmo de cualquier método estocástico.

In [ ]:
pp0 = ParamPart()
S_T1, T_T1 = 0.05, 60.0
St = S_T1 * T_T1
M1 = MallaTamanos(0.05, 10, k=1); j0 = M1.n - 3
N0 = np.zeros(M1.n); N0[j0] = 1000.0
_, Y1 = resolver_pbm(M1, pp0, 0, 1, 1e9, T_T1, {"erosion": False, "vaciado": False, "S_cte": S_T1}, N0=N0,
                     t_eval=np.array([0, T_T1]), rtol=1e-11)
G = 10
exacta = np.array([1000 * 2 ** g * np.exp(-St) * St ** g / math.factorial(g) for g in range(G)])
e_T1 = float(np.max(np.abs(Y1[-1, [j0 - g for g in range(G)]] - exacta)) / exacta.max())

poisson = np.array([np.exp(-St) * St ** g / math.factorial(g) for g in range(G)])
ns_mc, err_mc = [500, 2000, 8000], []
d0 = M1.d[j0]
for n in ns_mc:
    es = []
    for s in range(4):
        r = mc_lagrangiano(pp0, 1.0, 1, 1e9, T_T1, dt=0.05, n_part=n, semilla=s, d_ini=d0, guardar_cada=T_T1,
                           opciones={"erosion": False, "vaciado": False, "S_cte": S_T1}, n_max=10 ** 7)
        gen = np.rint(np.log(d0 / r["d_final"]) / np.log(2 ** (1 / 3))).astype(int)
        m = r["d_final"] ** 3
        frac = np.array([m[gen == g].sum() for g in range(G)]) / m.sum()
        es.append(np.abs(frac - poisson).sum())
    err_mc.append(np.mean(es))
p_mc = np.polyfit(np.log(ns_mc), np.log(err_mc), 1)[0]

fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 3.9))
gs = np.arange(G)
a1.bar(gs, poisson, color=GRIS_MEDIO, alpha=0.5, label="Poisson exacta")
a1.plot(gs, Y1[-1, [j0 - g for g in range(G)]] * (2.0 ** -gs) / 1000, "o", color=C1, label="balance de poblaciones")
a1.plot(gs, frac, "^", mfc="none", color=C2, label="Monte Carlo (N = 8000)")
a1.set(xlabel="generación (nº de roturas)", ylabel="fracción de masa", title=f"T1 · Rotura constante, St = {St:.0f}"); a1.legend(fontsize=9)
a2.loglog(ns_mc, err_mc, "o-", color=C2, label=f"Monte Carlo (pendiente {p_mc:.2f})")
a2.loglog(ns_mc, err_mc[0] * (np.array(ns_mc) / ns_mc[0]) ** -0.5, ":", color=GRIS_MEDIO, label="pendiente −1/2")
a2.minorticks_off(); a2.set_xticks(ns_mc); a2.set_xticklabels([str(n) for n in ns_mc])
a2.set(xlabel="partículas numéricas", ylabel="error L1 de la fracción de masa", title="Convergencia estadística"); a2.legend(fontsize=9)
fig.tight_layout(); plt.show()
VERIFICACION += [
    {"test": "T1 PBM rotura vs solución exacta", "valor": e_T1, "criterio": "< 1e-8", "pasa": e_T1 < 1e-8},
    {"test": "T1b Monte Carlo: error ∝ N^(−1/2)", "valor": p_mc, "criterio": "−0.75 < p < −0.3", "pasa": -0.75 < p_mc < -0.3},
]

### T2 — Erosión de partículas iguales: núcleo que se encoge
Con velocidad de retroceso constante $r$, el diámetro baja linealmente, $d(t) = d_0 - r t$. La masa sólida es $N_0\rho_p\pi(d_0 - rt)^3/6$. En el balance de poblaciones, la erosión es una advección en el espacio de tamaños con esquema *upwind*, así que el error debe ser de **orden 1** en el espaciado de la malla. Refinamos con $k$ = 1, 2, 4, 8 y 16 celdas por cada mitad de volumen. En el Monte Carlo la erosión es exacta, porque cada partícula guarda su diámetro.

### T3 — Conservación de la masa con todos los procesos
Sólido + disuelto + vaciado (sólido) + vaciado (disuelto) = masa ingerida, en cada instante.

In [ ]:
d0_T2, r_T2, T_T2 = 3.0, 0.01, 150.0
ks, e_T2 = [1, 2, 4, 8, 16], []
for k in ks:
    Mk = MallaTamanos(0.05, 10, k=k)
    j = int(np.argmin(np.abs(Mk.d - d0_T2))); N0 = np.zeros(Mk.n); N0[j] = 1.0
    t2, Y2 = resolver_pbm(Mk, pp0, 0, 1, 1e9, T_T2, {"rotura": False, "vaciado": False, "r_cte": r_T2}, N0=N0,
                          t_eval=np.linspace(0, T_T2, 61), rtol=1e-10)
    ms = masas_pbm(Y2, Mk, pp0)
    ex = pp0.rho_p * np.pi * np.clip(Mk.d[j] - r_T2 * t2, 0, None) ** 3 / 6
    e_T2.append(float(np.max(np.abs(ms["sólido"] - ex)) / ex[0]))
    if k in (1, 16):
        globals()[f"curva_T2_k{k}"] = (t2, ms["sólido"] / ex[0], ex / ex[0])
p_T2 = np.polyfit(np.log(ks), np.log(e_T2), 1)[0]

M3 = MallaTamanos(0.05, 10, k=3)
V0_c, Te_c, masa_c = 340.0, 255 / 2.0, 13.0           # comida de consenso: volumen, T_e (2 kcal/min), masa de clara
t3, Y3 = resolver_pbm(M3, pp0, masa_c, V0_c, Te_c, 240, t_eval=np.linspace(0, 240, 241))
ms3 = masas_pbm(Y3, M3, pp0)
total = ms3["sólido"] + ms3["disuelto"] + ms3["salida sólida"] + ms3["salida disuelta"]
e_T3 = float(np.max(np.abs(total - masa_c)) / masa_c)

fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 3.8))
for k, col in ((1, C1), (16, C3)):
    tt, num, ex = globals()[f"curva_T2_k{k}"]
    a1.plot(tt, num, color=col, label=f"PBM, k = {k}")
a1.plot(tt, ex, color=GRIS, lw=3, alpha=0.35, label="exacta")
a1.set(xlabel="t [min]", ylabel="masa sólida / inicial", title="T2 · Erosión de partículas de 3 mm"); a1.legend(fontsize=9)
a2.loglog(ks, e_T2, "o-", color=C1, label=f"PBM (pendiente {p_T2:.2f})")
a2.loglog(ks, e_T2[0] * (np.array(ks) / ks[0]) ** -1.0, ":", color=GRIS_MEDIO, label="pendiente −1")
a2.minorticks_off(); a2.set_xticks(ks); a2.set_xticklabels([str(k) for k in ks])
a2.set(xlabel="celdas por mitad de volumen (k)", ylabel="error máximo relativo", title="Convergencia de la erosión"); a2.legend(fontsize=9)
fig.tight_layout(); plt.show()
VERIFICACION += [
    {"test": "T2 erosión PBM: orden 1 en la malla de tamaños", "valor": -p_T2, "criterio": "|p − 1| < 0.15", "pasa": abs(-p_T2 - 1) < 0.15},
    {"test": "T3 conservación de masa (todos los procesos)", "valor": e_T3, "criterio": "< 1e-10", "pasa": e_T3 < 1e-10},
]

### T4 — Arrastre y sedimentación de una partícula
En fluido en reposo y con Reynolds de partícula pequeño, una partícula que parte del reposo cae con
$$ y(t) = y_0 - v_t\big[t - \tau_p (1 - e^{-t/\tau_p})\big]. $$
El integrador exponencial la reproduce exactamente si $\tau_p$ es constante. Aquí la única diferencia es la pequeñísima corrección de Schiller–Naumann a $\mathrm{Re}_p \sim 10^{-5}$, del orden de $10^{-4}$. Con una partícula de 3 mm, $\mathrm{Re}_p$ ya no es pequeño: la velocidad terminal debe cumplir el punto fijo $v_t = \tau_p(v_t)\,(1-\rho_f/\rho_p)\,g$ con la corrección de Schiller–Naumann.

In [ ]:
def campo_reposo(Lx=10.0, Ly=20.0, dx=0.1):
    x = np.arange(0, Lx, dx); y = np.arange(0, Ly, dx)
    z = np.zeros((2, y.size, x.size))
    return CampoPeriodico({"t": np.array([0.0, 1.0]), "ux": z, "uy": z, "solido": np.zeros((2, y.size, x.size), bool),
                           "x": x, "y": y, "dx": dx, "periodo": 1.0, "nu": 0.1})

cr = campo_reposo()
# (a) partícula pequeña: régimen de Stokes
d_p, rho_p, g_ = 0.05, 1.02, 981.0             # mm, g/cm³
tau_s = rho_p * (d_p / 10) ** 2 / (18 * 1.0 * 0.1)
vt_s = tau_s * (1 - 1 / rho_p) * g_
rs = seguir_particulas(cr, np.array([d_p]), rho_p, t_final=0.02, dt=1e-4, guardar_cada=1e-4)
ys = rs["y"][:, 0]; t_s = rs["t"]
exacta_y = ys[0] - vt_s * (t_s - tau_s * (1 - np.exp(-t_s / tau_s)))
e_T4 = float(np.max(np.abs(ys - exacta_y)) / (vt_s * t_s[-1]))
# (b) partícula de 3 mm: velocidad terminal con Schiller–Naumann
d_b, rho_b = 3.0, 1.03
rb = seguir_particulas(cr, np.array([d_b]), rho_b, t_final=1.0, dt=1e-3, guardar_cada=0.01)
v_sim = -(rb["y"][-1, 0] - rb["y"][-2, 0]) / (rb["t"][-1] - rb["t"][-2])
v = 1.0
for _ in range(100):
    v = tiempo_relajacion(d_b / 10, rho_b, 1.0, 0.1, v) * (1 - 1 / rho_b) * g_
e_T4b = abs(v_sim - v) / v
print(f"Stokes (d = {d_p} mm): error relativo de la trayectoria = {e_T4:.1e}")
print(f"Schiller–Naumann (d = 3 mm, ρp = {rho_b}): v_t simulada = {v_sim:.3f} cm/s, punto fijo = {v:.3f} cm/s,"
      f" Re_p = {v * 0.3 / 0.1:.1f}")
VERIFICACION += [
    {"test": "T4 sedimentación de Stokes", "valor": e_T4, "criterio": "< 1e-3", "pasa": e_T4 < 1e-3},
    {"test": "T4b velocidad terminal Schiller–Naumann", "valor": e_T4b, "criterio": "< 1e-3", "pasa": e_T4b < 1e-3},
]

### T5 — Monte Carlo frente a balance de poblaciones, con todos los procesos
Las dos implementaciones resuelven las mismas leyes: rotura con debilitamiento por ácido, erosión dependiente del pH y criba. Se calculan de forma independiente: el PBM es determinista sobre una malla de tamaños y el Monte Carlo es estocástico con tamaños continuos. Las curvas de retención deben coincidir dentro del ruido estadístico más el error de la malla de tamaños del PBM.

In [ ]:
t0 = time.time()
mcs = [mc_lagrangiano(pp0, masa_c, V0_c, Te_c, 240, dt=0.1, n_part=10000, semilla=s, n_max=100000) for s in (1, 2)]
mc5 = mcs[0]
ret_mc_ind = np.array([100 * (m["sólido"] + m["disuelto"]) / masa_c for m in mcs])
ret_mc = ret_mc_ind.mean(axis=0)
ret_pbm = 100 * (ms3["sólido"] + ms3["disuelto"]) / masa_c
dif_T5 = float(np.max(np.abs(np.interp(mc5["t"], t3, ret_pbm) - ret_mc)))
print(f"Monte Carlo (2 réplicas): {time.time() - t0:.1f} s, hasta {max(m['n'].max() for m in mcs)} partículas numéricas;"
      f" diferencia entre réplicas: {np.max(np.abs(ret_mc_ind[0] - ret_mc_ind[1])):.1f} puntos")
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 3.8))
a1.plot(t3, ret_pbm, color=C1, label="balance de poblaciones (k = 3)")
a1.plot(mc5["t"], ret_mc, color=C2, lw=1.2, ls="--", label="Monte Carlo (media de 2 × 10 000)")
a1.set(xlabel="t [min]", ylabel="retención [%]", title="T5 · Retención (parámetros nominales)"); a1.legend(fontsize=9)
a2.plot(t3, [d50_masa(Y3[i, :M3.n], M3, pp0) for i in range(len(t3))], color=C1, label="PBM")
a2.plot(mc5["t"], mc5["d50"], color=C2, lw=1.2, ls="--", label="Monte Carlo")
a2.axhline(pp0.d_c, color=GRIS_MEDIO, lw=0.8, ls=":"); a2.text(235, pp0.d_c + 0.05, "corte del píloro", ha="right", fontsize=9, color=TINTA2)
a2.set(xlabel="t [min]", ylabel="d50 en masa [mm]", title="Tamaño mediano de lo que queda"); a2.legend(fontsize=9)
fig.tight_layout(); plt.show()
VERIFICACION.append({"test": "T5 Monte Carlo vs PBM (retención)", "valor": dif_T5, "criterio": "< 3 puntos %", "pasa": dif_T5 < 3})

**Una advertencia sobre el tamaño mediano.** Al final quedan pocas partículas grandes. El d50 del Monte Carlo es ruidoso y el del PBM está suavizado por la difusión numérica de la malla de tamaños. Por eso la comparación cuantitativa se hace con la retención, que es una magnitud integral.

## 4. Partículas en el flujo del antro (fase 3)

Cargamos el campo de velocidades de un periodo de onda de la fase 3. Si no está, lo regeneramos con el mismo código LBM.

In [ ]:
def obtener_campo():
    if os.path.exists("campo_antro.npz"):
        datos = dict(np.load("campo_antro.npz"))
        if "tau_max" in datos:
            print("Usando campo_antro.npz de la fase 3")
            return datos
        print("campo_antro.npz no incluye tensiones: se regenera")
    print("Regenerando el flujo del antro con Lattice Boltzmann (≈ 75 s en CPU)...")
    exec(CODIGO_LBM, globals())
    pa = ParamAntro(); R = Red(np)
    S = simular_antro(pa, R, periodos=2, n_particulas=10, informar=False)
    t = S["t"]; k = int(round(pa.T / (t[1] - t[0]))); ult = slice(len(t) - k - 1, len(t))
    dx = S["U"].dx
    datos = dict(t=t[ult] - t[ult][0], ux=np.nan_to_num(S["ux"][ult]), uy=np.nan_to_num(S["uy"][ult]),
                 solido=S["solido"][ult], x=(np.arange(S["nx"]) - 0.5) * dx, y=(np.arange(S["ny"]) + 0.5) * dx,
                 dx=dx, periodo=pa.T, nu=pa.nu, tau_max=np.nan_to_num(S["tau_max"][ult]))
    np.savez_compressed("campo_antro.npz", **datos)
    return datos

In [ ]:
CODIGO_LBM = r'''
# ===========================================================================
# Lattice Boltzmann D2Q9 escrito con funciones PURAS (sin modificar arrays en el sitio).
# Así el mismo código corre con NumPy, CuPy (GPU, misma API) o jax.numpy (GPU + jit).
# Unidades de red: dx = dt = 1, c_s^2 = 1/3, nu = (tau - 1/2)/3.
# Convenio de arrays: f tiene forma (9, ny, nx); la fila j = 0 es el fondo (y crece hacia arriba).
# ===========================================================================
CX  = [0, 1, 0, -1, 0, 1, -1, -1, 1]
CY  = [0, 0, 1, 0, -1, 1, 1, -1, -1]
W   = [4 / 9] + [1 / 9] * 4 + [1 / 36] * 4
OPP = [0, 3, 4, 1, 2, 7, 8, 5, 6]          # dirección opuesta


class Red:
    # Constantes D2Q9 ya convertidas al backend (xp) y al tipo de dato elegidos
    def __init__(self, xp, dtype="float64"):
        self.xp, self.dtype = xp, dtype
        self.cx = xp.asarray(CX, dtype=dtype).reshape(9, 1, 1)
        self.cy = xp.asarray(CY, dtype=dtype).reshape(9, 1, 1)
        self.w = xp.asarray(W, dtype=dtype).reshape(9, 1, 1)


def equilibrio(rho, ux, uy, R):
    cu = 3.0 * (R.cx * ux + R.cy * uy)
    usq = 1.5 * (ux * ux + uy * uy)
    return R.w * rho * (1.0 + cu + 0.5 * cu * cu - usq)


def macroscopicas(f, R, Fx=0.0, Fy=0.0):
    rho = f.sum(axis=0)
    jx = f[1] - f[3] + f[5] - f[6] - f[7] + f[8]        # sum_i f_i c_ix, escrito explícitamente (más rápido)
    jy = f[2] - f[4] + f[5] + f[6] - f[7] - f[8]
    ux = (jx + 0.5 * Fx) / rho                          # desplazamiento de Guo: u = (sum f c + F/2)/rho
    uy = (jy + 0.5 * Fy) / rho
    return rho, ux, uy


def colision_bgk(f, rho, ux, uy, tau, R, Fx=0.0, Fy=0.0):
    feq = equilibrio(rho, ux, uy, R)
    cu = R.cx * ux + R.cy * uy
    cF = R.cx * Fx + R.cy * Fy
    # término de fuerza de Guo (2002): segundo orden con fuerza volumétrica
    S = (1.0 - 0.5 / tau) * R.w * (3.0 * ((R.cx - ux) * Fx + (R.cy - uy) * Fy) + 9.0 * cu * cF)
    return f - (f - feq) / tau + S


def propagar(f, R):
    xp = R.xp
    return xp.stack([xp.roll(f[i], shift=(CY[i], CX[i]), axis=(0, 1)) for i in range(9)])


def rebote(fs, fpost, solido, uwx, uwy, rho, R):
    # Rebote a mitad de enlace con pared móvil (Ladd):
    # si la población i llega a un nodo fluido x desde un nodo sólido x - c_i, se sustituye por
    # f_i(x) = f*_opp(i)(x) + 6 w_i rho (c_i · u_w),  con u_w la velocidad de la pared en ese nodo sólido.
    xp = R.xp
    salida = []
    fluido = ~solido
    for i in range(9):
        if i == 0:
            salida.append(fs[0]); continue
        desde_solido = xp.roll(solido, shift=(CY[i], CX[i]), axis=(0, 1)) & fluido
        cu_w = 0.0
        if CX[i] != 0 and uwx is not None:
            cu_w = cu_w + CX[i] * xp.roll(uwx, shift=(CY[i], CX[i]), axis=(0, 1))
        if CY[i] != 0 and uwy is not None:
            cu_w = cu_w + CY[i] * xp.roll(uwy, shift=(CY[i], CX[i]), axis=(0, 1))
        rebotado = fpost[OPP[i]] + 6.0 * W[i] * rho * cu_w
        salida.append(xp.where(desde_solido, rebotado, fs[i]))
    return xp.stack(salida)


def rellenar(f, solido_viejo, solido_nuevo, rho, uwx, uwy, R):
    # Nodos que pasan de sólido a fluido ("nodos frescos") se inicializan con el equilibrio:
    # densidad = media de los vecinos fluidos, velocidad = la de la pared (Lallemand y Luo, 2003).
    xp = R.xp
    fresco = solido_viejo & ~solido_nuevo
    fluido_previo = (~solido_viejo).astype(rho.dtype)
    suma, n = 0.0 * rho, 0.0 * rho
    for i in range(1, 9):
        suma = suma + xp.roll(rho * fluido_previo, shift=(CY[i], CX[i]), axis=(0, 1))
        n = n + xp.roll(fluido_previo, shift=(CY[i], CX[i]), axis=(0, 1))
    rho_vec = xp.where(n > 0, suma / xp.maximum(n, 1.0), 1.0)
    cero = 0.0 * rho
    f_fresco = equilibrio(rho_vec, cero if uwx is None else uwx, cero if uwy is None else uwy, R)
    f = xp.where(fresco[None], f_fresco, f)
    # dentro del sólido guardamos el equilibrio en reposo, f_i = w_i (no interviene en la dinámica)
    return xp.where(solido_nuevo[None], R.w + 0.0 * f, f)


def crear_paso(R, tau, geometria, Fx=0.0, Fy=0.0):
    # Devuelve paso(f, t) -> f(t+1). geometria(t) -> (solido, uwx, uwy) en unidades de red.
    def paso(f, t):
        solido, uwx, uwy = geometria(t)
        rho, ux, uy = macroscopicas(f, R, Fx, Fy)
        fpost = colision_bgk(f, rho, ux, uy, tau, R, Fx, Fy)
        fs = propagar(fpost, R)
        fs = rebote(fs, fpost, solido, uwx, uwy, rho, R)
        solido2, uwx2, uwy2 = geometria(t + 1)
        return rellenar(fs, solido, solido2, rho, uwx2, uwy2, R)
    return paso


def tension_viscosa(f, tau, R, Fx=0.0, Fy=0.0):
    # Tensor de tensiones viscosas a partir de la parte de no equilibrio (sin derivar el campo):
    # sigma_ab = -(1 - 1/(2 tau)) sum_i (f_i - feq_i) c_ia c_ib    [unidades de red]
    rho, ux, uy = macroscopicas(f, R, Fx, Fy)
    fneq = f - equilibrio(rho, ux, uy, R)
    k = -(1.0 - 0.5 / tau)
    sxx = k * (fneq * R.cx * R.cx).sum(axis=0)
    syy = k * (fneq * R.cy * R.cy).sum(axis=0)
    sxy = k * (fneq * R.cx * R.cy).sum(axis=0)
    return sxx, syy, sxy


def geometria_fija(solido, R):
    xp = R.xp
    cero = xp.zeros(solido.shape, dtype=R.dtype)
    return lambda t: (solido, cero, cero)


# ---------------------------------------------------------------------------
# Paredes que se mueven: canal peristáltico periódico (test de Shapiro)
# ---------------------------------------------------------------------------
def geometria_peristaltica(nx, ny, a, phi, lam, c, R):
    # Semiancho H(x,t) = a (1 + phi sin(2 pi (x - c t)/lam)) alrededor del eje y = ny/2.
    # Las paredes se mueven solo en vertical (como en Shapiro et al., 1969).
    xp = R.xp
    x = xp.arange(nx, dtype=R.dtype)[None, :] + 0.5
    y = xp.arange(ny, dtype=R.dtype)[:, None] + 0.5
    yc = ny / 2.0
    k = 2 * math.pi / lam
    def geom(t):
        H = a * (1.0 + phi * xp.sin(k * (x - c * t)))
        dHdt = -a * phi * k * c * xp.cos(k * (x - c * t))
        abajo, arriba = y < yc - H, y > yc + H
        solido = abajo | arriba
        uwy = xp.where(abajo, -dHdt, 0.0 * y) + xp.where(arriba, dHdt, 0.0 * y)
        return solido, None, uwy                        # uwx = None: las paredes solo se mueven en vertical
    return geom


def integrar(paso, f, t0, n_pasos, R, usar_jax=False):
    # Bucle de tiempo. Con JAX se compila con lax.fori_loop (todo el bucle en la GPU).
    if usar_jax:
        from jax import lax
        f, _ = lax.fori_loop(0, n_pasos, lambda i, ft: (paso(ft[0], ft[1]), ft[1] + 1.0), (f, R.xp.asarray(t0, dtype=R.dtype)))
        return f
    for n in range(n_pasos):
        f = paso(f, t0 + n)
    return f

@dataclass
class ParamAntro:
    # Geometría y movimiento (unidades físicas: cm, s)
    Lx: float = 10.0          # cm   longitud (fondo -> píloro)
    H: float = 4.0            # cm   anchura en reposo
    margen: float = 1.0       # cm   holgura para que el fondo pueda dilatarse
    x_ini: float = 4.5        # cm   donde empiezan las ondas antrales
    x_plena: float = 6.5      # cm   donde alcanzan su amplitud máxima
    L_fondo: float = 3.5      # cm   zona proximal que se dilata para conservar el volumen
    A_max: float = 1.4        # cm   profundidad de la contracción en cada pared (70 % de oclusión)
    w: float = 0.5            # cm   semiancho de la onda
    c: float = 0.25           # cm/s velocidad de la onda peristáltica
    T: float = 20.0           # s    periodo (3 ondas por minuto)
    nu: float = 0.1           # cm²/s viscosidad cinemática (comida líquida espesa; agua = 0.01)
    rho: float = 1.0          # g/cm³
    # Discretización
    nx: int = 128             # nodos a lo largo (sin contar las paredes de los extremos)
    tau: float = 0.65


class Unidades:
    # Conversión física <-> red: dx fija la malla; dt sale de igualar la viscosidad.
    def __init__(self, pa):
        self.dx = pa.Lx / pa.nx                        # cm
        self.nu_red = (pa.tau - 0.5) / 3.0
        self.dt = self.nu_red * self.dx ** 2 / pa.nu    # s
        self.vel = self.dx / self.dt                    # (cm/s) por unidad de red
        self.tension = pa.rho * self.vel ** 2           # (dyn/cm² = 0.1 Pa) por unidad de red

    def resumen(self, pa):
        return {"dx [cm]": self.dx, "dt [s]": self.dt, "pasos por onda": pa.T / self.dt,
                "c onda [red]": pa.c / self.vel, "Re = c (H/2)/nu": pa.c * (pa.H / 2) / pa.nu,
                "nu [red]": self.nu_red}


def geometria_antro(pa, R):
    # Paredes superior e inferior con ondas que viajan hacia el píloro (x = Lx).
    # El fondo (x < L_fondo) se dilata lo justo para que el volumen total no cambie.
    # Extremos x = 0 (fondo) y x = Lx (píloro, cerrado en esta fase) son paredes fijas.
    xp = R.xp
    U = Unidades(pa)
    dx = U.dx
    ny = int(round((pa.H + 2 * pa.margen) / dx))
    nx = pa.nx + 2                                       # + una columna de pared en cada extremo
    x = (xp.arange(nx, dtype=R.dtype)[None, :] - 0.5) * dx          # cm (la columna 0 es pared)
    y = (xp.arange(ny, dtype=R.dtype)[:, None] + 0.5) * dx
    rampa = xp.clip((x - pa.x_ini) / (pa.x_plena - pa.x_ini), 0.0, 1.0)
    envolvente = pa.A_max * rampa * rampa * (3 - 2 * rampa)         # suave (smoothstep)
    ventana_fondo = xp.clip((pa.L_fondo + 1.0 - x) / 1.0, 0.0, 1.0) * (x > 0) * (x < pa.Lx)
    int_ventana = float(xp.sum(ventana_fondo)) * dx
    x0 = pa.x_ini - 3 * pa.w
    K = int(math.ceil((pa.Lx + 3 * pa.w - x0) / (pa.c * pa.T))) + 1
    extremos = (x <= 0) | (x >= pa.Lx)
    def ondas(t_fis):
        s = t_fis % pa.T
        d = 0.0 * x
        dd = 0.0 * x
        for k in range(K):
            xk = x0 + pa.c * (s + k * pa.T)
            g = xp.exp(-((x - xk) / pa.w) ** 2)
            d = d + g
            dd = dd + g * 2 * (x - xk) * pa.c / pa.w ** 2
        return envolvente * d, envolvente * dd                      # delta [cm], d delta/dt [cm/s]
    def geom(t):
        t_fis = t * U.dt
        delta, ddelta = ondas(t_fis)
        interior = (x > 0) & (x < pa.Lx)
        area = xp.sum(delta * interior) * dx                         # área contraída por pared
        darea = xp.sum(ddelta * interior) * dx
        d_f = area / int_ventana                                     # dilatación del fondo
        dd_f = darea / int_ventana
        yb = pa.margen + delta - ventana_fondo * d_f
        yt = pa.margen + pa.H - delta + ventana_fondo * d_f
        vb = ddelta - ventana_fondo * dd_f                           # cm/s
        vt = -ddelta + ventana_fondo * dd_f
        abajo, arriba = y < yb, y > yt
        solido = abajo | arriba | extremos
        uwy = (xp.where(abajo, vb, 0.0 * y) + xp.where(arriba, vt, 0.0 * y)) * ~extremos / U.vel
        return solido, None, uwy
    return geom, nx, ny, U

def a_numpy(a):
    # Convierte un array de cualquier backend (NumPy, CuPy, JAX) a NumPy
    if hasattr(a, "get"):
        return a.get()              # CuPy
    return np.asarray(a)            # NumPy y JAX


def interpolar_bilineal(campo, xs, ys):
    # campo (ny, nx) en nodos de red; xs, ys en coordenadas de red (centro del nodo i en x = i)
    ny, nx = campo.shape
    x = np.clip(xs, 0, nx - 1.001); y = np.clip(ys, 0, ny - 1.001)
    i0 = np.floor(x).astype(int); j0 = np.floor(y).astype(int)
    tx = x - i0; ty = y - j0
    c00 = campo[j0, i0]; c10 = campo[j0, i0 + 1]; c01 = campo[j0 + 1, i0]; c11 = campo[j0 + 1, i0 + 1]
    return (1 - tx) * (1 - ty) * c00 + tx * (1 - ty) * c10 + (1 - tx) * ty * c01 + tx * ty * c11


def simular_antro(pa, R, periodos=4, guardar_cada_s=0.25, n_particulas=3000, semilla=1,
                  usar_jax=False, informar=True):
    # Ejecuta el antro y devuelve instantáneas (en NumPy y unidades físicas) + trayectorias de trazadores.
    xp = R.xp
    geom, nx, ny, U = geometria_antro(pa, R)
    paso = crear_paso(R, pa.tau, geom)
    f = R.w * xp.ones((9, ny, nx), dtype=R.dtype)                     # reposo, rho = 1
    pasos_bloque = max(1, int(round(guardar_cada_s / U.dt)))
    n_bloques = int(round(periodos * pa.T / (pasos_bloque * U.dt)))
    if usar_jax:
        import jax
        avanzar = jax.jit(lambda f, t0: integrar(paso, f, t0, pasos_bloque, R, usar_jax=True))
    else:
        avanzar = lambda f, t0: integrar(paso, f, t0, pasos_bloque, R)
    # trazadores: repartidos al azar por el fluido inicial (coordenadas de red)
    rng = np.random.default_rng(semilla)
    s0 = a_numpy(geom(0)[0])
    cand = np.argwhere(~s0)
    elegido = cand[rng.choice(len(cand), n_particulas, replace=False)]
    py = elegido[:, 0] + rng.uniform(-0.5, 0.5, n_particulas)
    px = elegido[:, 1] + rng.uniform(-0.5, 0.5, n_particulas)
    hist = {k: [] for k in ("t", "ux", "uy", "rho", "solido", "tau_max", "px", "py", "masa")}
    def campos(f, t):
        rho, ux, uy = macroscopicas(f, R)
        sxx, syy, sxy = tension_viscosa(f, pa.tau, R)
        tmax = xp.sqrt(0.25 * (sxx - syy) ** 2 + sxy ** 2)            # cizalla máxima
        sol = geom(t)[0]
        return [a_numpy(v) for v in (rho, ux, uy, tmax, sol)]
    t_red = 0
    rho, ux, uy, tmax, sol = campos(f, t_red)
    t_ini = time.time()
    for b in range(n_bloques + 1):
        hist["t"].append(t_red * U.dt)
        hist["ux"].append(np.where(sol, np.nan, ux) * U.vel)
        hist["uy"].append(np.where(sol, np.nan, uy) * U.vel)
        hist["rho"].append(np.where(sol, np.nan, rho))
        hist["tau_max"].append(np.where(sol, np.nan, tmax) * U.tension / 10.0)   # Pa
        hist["solido"].append(sol)
        hist["px"].append(px.copy()); hist["py"].append(py.copy())
        hist["masa"].append(float(np.sum(np.where(sol, 0.0, rho))))
        if b == n_bloques:
            break
        f = avanzar(f, float(t_red))
        t_red += pasos_bloque
        rho2, ux2, uy2, tmax2, sol2 = campos(f, t_red)
        # trazadores: Heun con el campo al principio y al final del bloque (desplazamiento en nodos)
        n_p = pasos_bloque
        vx1 = interpolar_bilineal(np.where(sol, 0, ux), px, py); vy1 = interpolar_bilineal(np.where(sol, 0, uy), px, py)
        qx, qy = px + n_p * vx1, py + n_p * vy1
        vx2 = interpolar_bilineal(np.where(sol2, 0, ux2), qx, qy); vy2 = interpolar_bilineal(np.where(sol2, 0, uy2), qx, qy)
        nx_, ny_ = px + 0.5 * n_p * (vx1 + vx2), py + 0.5 * n_p * (vy1 + vy2)
        ii = np.clip(np.rint(nx_).astype(int), 0, nx - 1); jj = np.clip(np.rint(ny_).astype(int), 0, ny - 1)
        dentro = ~sol2[jj, ii]
        px = np.where(dentro, nx_, px); py = np.where(dentro, ny_, py)   # si caería en la pared, se queda
        rho, ux, uy, tmax, sol = rho2, ux2, uy2, tmax2, sol2
        if informar and (b + 1) % max(1, n_bloques // 8) == 0:
            print(f"  t = {t_red * U.dt:6.1f} s   ({time.time() - t_ini:5.1f} s de cálculo)")
    out = {k: np.array(v) for k, v in hist.items()}
    out["U"], out["nx"], out["ny"], out["pasos"] = U, nx, ny, t_red
    out["segundos"] = time.time() - t_ini
    return out



'''

In [ ]:
CAMPO = CampoPeriodico(obtener_campo())
tau_hidro = float(np.max(CAMPO.tau)) if CAMPO.tau is not None else np.nan
d_ref = 3e-3                                           # m
tension_contacto = (0.2 / d_ref ** 2, 2.0 / d_ref ** 2)
print(f"Cizalla hidrodinámica máxima en el antro (fase 3): {tau_hidro:.3f} Pa")
print(f"Tensión de contacto sobre una partícula de 3 mm (0.2–2 N): {tension_contacto[0]/1e3:.0f}–{tension_contacto[1]/1e3:.0f} kPa")
print(f"Cociente: ~{tension_contacto[0] / tau_hidro:.0e}  ->  la rotura la producen las paredes, no el fluido")

### 4.1 Frecuencia de molienda $f_{TA}(d)$
Seguimos 1500 partículas de cinco tamaños (0.5–4 mm) durante 10 ondas (200 s). Su densidad es de 1.03 g/cm³, ligeramente más densas que el quimo. Contamos cuántas veces por minuto cada partícula entra en un **cuello de contracción del antro terminal**: $x > 8$ cm, con hueco local menor que el 60 % del calibre en reposo. Esa es la $f_{TA}(d)$ que alimenta la función de selección de rotura. También repetimos con partículas de la misma densidad que el fluido, para ver el papel de la sedimentación.

In [ ]:
tamanos = [0.5, 1.0, 2.0, 3.0, 4.0]
d_lag = np.repeat(tamanos, 300)
t0 = time.time()
LAG = {rp: seguir_particulas(CAMPO, d_lag, rp, t_final=200.0, dt=0.02) for rp in (1.03, 1.0)}
print(f"Seguimiento de partículas: {time.time() - t0:.0f} s")
F_TA = {rp: np.array([L["eventos_por_min"][L["d_mm"] == dv].mean() for dv in tamanos]) for rp, L in LAG.items()}
display(pd.DataFrame({"d [mm]": tamanos, "f_TA (ρp = 1.03) [1/min]": F_TA[1.03], "f_TA (ρp = 1.00) [1/min]": F_TA[1.0],
                      "altura media final (ρp = 1.03) [cm]": [LAG[1.03]["y"][-1][LAG[1.03]["d_mm"] == dv].mean() for dv in tamanos]}
                     ).style.format(precision=3).hide(axis="index"))

fig, axs = plt.subplots(1, 2, figsize=(13, 3.6))
cols = dict(zip(tamanos, [C1, C3, C4, C2, C5]))
for a, rp in zip(axs, (1.0, 1.03)):
    L = LAG[rp]
    k_fin = len(L["t"]) - 1
    sol = CAMPO.sol[CAMPO._marco(L["t"][k_fin])[0]]
    a.contour(CAMPO.x, CAMPO.y, sol.astype(float), levels=[0.5], colors=GRIS, linewidths=0.8)
    for dv in (0.5, 3.0):
        m = np.where(L["d_mm"] == dv)[0][:40]
        for q in m:
            a.plot(L["x"][-60:, q], L["y"][-60:, q], color=cols[dv], lw=0.6, alpha=0.6)
        a.scatter(L["x"][-1, m], L["y"][-1, m], s=6 * dv, color=cols[dv], label=f"{dv} mm")
    a.axvline(8.0, color=GRIS_MEDIO, lw=0.8, ls=":")
    a.set(xlim=(0, 10), ylim=(0, CAMPO.y[-1]), aspect="equal", xlabel="x [cm]", title=f"ρp = {rp} g/cm³: últimas trayectorias (30 s)"); a.grid(False)
axs[0].set_ylabel("y [cm]"); axs[1].legend(loc="upper left", fontsize=9)
fig.tight_layout(); plt.show()

**Lectura.** Con solo un 3 % más de densidad que el quimo, las partículas sedimentan. Su velocidad terminal es comparable a la del flujo, y se acumulan junto a la pared inferior. Allí las barren las contracciones, de modo que la frecuencia de visitas al antro terminal **apenas depende del tamaño**, con unas 0.4–0.5 visitas por minuto. Es del orden de 1 visita cada 6 contracciones.

## 5. Validación: la comida de consenso con partículas y criba

Usamos $f_{TA}(d)$ de la sección 4. Los demás parámetros se fijan **a priori**, sin ajustarlos a los datos de retención, y su incertidumbre se recorre con un Monte Carlo de hipercubo latino (rangos en la tabla). Los datos de validación son los mismos que en la fase 1b: retención del 37–90 % a 1 h, del 30–60 % a 2 h y del 0–10 % a 4 h.

In [ ]:
d_tab = np.array(tamanos); f_tab = F_TA[1.03]
PP = replace(ParamPart(), f_TA=(d_tab, f_tab))
RANGOS = [  # (parámetro, mínimo, máximo, escala, origen)
    ("d50_ini",    2.0,   4.0,   "lin", "masticación: mediana hasta ~3 mm"),
    ("escala_fTA", 0.5,   1.5,   "lin", "umbral de 'cuello' y densidad (sección 4)"),
    ("q_atrapada", 0.2,   0.8,   "lin", "supuesto"),
    ("g_min",      0.2,   1.0,   "lin", "supuesto: hueco de la oclusión terminal"),
    ("sigma_c0",   3e4,   3e5,   "log", "supuesto: geles proteicos / carne cocinada"),
    ("X_w",        30.0,  120.0, "lin", "supuesto"),
    ("r_e0",       1e-3,  1e-2,  "log", "supuesto: 1–10 µm/min"),
    ("d_c",        1.0,   2.0,   "lin", "criba pilórica < 1–2 mm"),
    ("kcal_min",   1.0,   4.0,   "log", "vaciado energético (fase 1b)"),
]
display(pd.DataFrame([{"parámetro": a, "mín": b, "máx": c, "escala": d, "origen": e} for a, b, c, d, e in RANGOS]).style.hide(axis="index"))

def aplicar(pp, muestra):
    cambios = {k: v for k, v in muestra.items() if k != "escala_fTA"}
    if "escala_fTA" in muestra:
        cambios["f_TA"] = (d_tab, f_tab * muestra["escala_fTA"])
    return replace(pp, **cambios)

def muestras(n, semilla=3):
    u = qmc.LatinHypercube(d=len(RANGOS), seed=semilla).random(n)
    out = []
    for fila in u:
        out.append({nom: (lo * (hi / lo) ** x if esc == "log" else lo + (hi - lo) * x) for (nom, lo, hi, esc, _), x in zip(RANGOS, fila)})
    return out

def simular_comida(pp, masa_g, V0, kcal, t_final=240, k=3, rtol=1e-8, debilitamiento=True):
    M = MallaTamanos(0.05, 10, k=k)
    T_e = kcal / pp.kcal_min
    t, Y = resolver_pbm(M, pp, masa_g, V0, T_e, t_final, opciones={"debilitamiento": debilitamiento},
                        t_eval=np.linspace(0, t_final, t_final + 1), rtol=rtol)
    ms = masas_pbm(Y, M, pp)
    return t, 100 * (ms["sólido"] + ms["disuelto"]) / masa_g, Y, M

def metricas_ret(t, ret):
    return {"retención 1 h [%]": float(np.interp(60, t, ret)), "retención 2 h [%]": float(np.interp(120, t, ret)),
            "retención 4 h [%]": float(np.interp(240, t, ret)),
            "T50 [min]": float(np.interp(50, ret[::-1], t[::-1])) if ret.min() < 50 else np.nan}

N_MC = 120
t0 = time.time()
t_nom, ret_nom, Y_nom, M_nom = simular_comida(PP, masa_c, V0_c, 255)
t_sd, ret_sd, _, _ = simular_comida(PP, masa_c, V0_c, 255, debilitamiento=False)
curvas, filas = [], []
for mu in muestras(N_MC):
    p_ = aplicar(PP, mu)
    tt, rr, _, _ = simular_comida(p_, masa_c, V0_c, 255)
    curvas.append(rr); filas.append(metricas_ret(tt, rr))
MCV = pd.DataFrame(filas); curvas = np.array(curvas)
print(f"Monte Carlo paramétrico ({N_MC} simulaciones): {time.time() - t0:.0f} s")

In [ ]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(13, 4.4), gridspec_kw={"width_ratios": [1.2, 1]})
lo, hi = np.percentile(curvas, [5, 95], axis=0)
a1.fill_between(t_nom, lo, hi, color=C1, alpha=0.18, lw=0, label="fase 4: percentiles 5–95")
a1.plot(t_nom, ret_nom, color=C1, label="fase 4: partículas + criba (nominal)")
a1.plot(t_sd, ret_sd, color=C1, ls=":", lw=1.4, label="fase 4 sin debilitamiento ácido")
a1.plot([60, 120, 240], [32.1, 1.9, 0.3], "s", color=C2, ms=7, label="fase 1b (nominal)")
for tc, (lo_, hi_) in zip((60, 120, 240), ((37, 90), (30, 60), (0, 10))):
    a1.plot([tc, tc], [lo_, hi_], color=GRIS, lw=6, alpha=0.35, solid_capstyle="butt")
a1.plot([], [], color=GRIS, lw=6, alpha=0.35, label="rango normal medido")
a1.set(xlabel="t [min]", ylabel="retención del marcador [%]", title="Comida de consenso: modelo frente a gammagrafía", ylim=(0, 105))
a1.legend(fontsize=8.5, loc="upper right")
# evolución de la distribución de tamaños (masa por celda)
m_cel = PP.rho_p * np.pi * M_nom.d ** 3 / 6
masa_cel = Y_nom[:, :M_nom.n] * m_cel
im = a2.imshow((masa_cel / masa_c * 100).T, origin="lower", aspect="auto", cmap="viridis",
               extent=[t_nom[0], t_nom[-1], 0, M_nom.n], vmin=0, vmax=np.percentile(masa_cel / masa_c * 100, 99.5))
ticks = [0.1, 0.5, 1, 2, 4, 8]
a2.set_yticks([np.interp(np.log(v), np.log(M_nom.d), np.arange(M_nom.n)) for v in ticks]); a2.set_yticklabels([str(v) for v in ticks])
a2.axhline(np.interp(np.log(PP.d_c), np.log(M_nom.d), np.arange(M_nom.n)), color="white", lw=1, ls="--")
a2.text(5, np.interp(np.log(PP.d_c), np.log(M_nom.d), np.arange(M_nom.n)) + 0.6, "corte del píloro", color="white", fontsize=9)
plt.colorbar(im, ax=a2, label="% de la masa ingerida")
a2.set(xlabel="t [min]", ylabel="tamaño de partícula [mm]", title="Distribución de tamaños en el estómago (nominal)"); a2.grid(False)
fig.tight_layout(); plt.show()
print(f"Retención a 2 h: con debilitamiento ácido {np.interp(120, t_nom, ret_nom):.1f} %, sin él {np.interp(120, t_sd, ret_sd):.1f} %")
grandes = Y_nom[-1, :M_nom.n] @ (m_cel * (M_nom.d > 6)) / masa_c * 100
print(f"Masa en partículas > 6 mm que sigue en el estómago a las 4 h: {grandes:.1f} % de lo ingerido")

### 5.1 Error numérico, paramétrico y de modelo, y registro de validación
- **Error numérico**: diferencia entre la configuración de producción (malla de tamaños con $k = 3$, `rtol = 1e-8`) y una más fina ($k = 12$, `rtol = 1e-11`). Aquí incluye el error de la malla de tamaños, que en T2 vimos que es de orden 1.
- **Error paramétrico**: semiancho de la banda 5–95 del Monte Carlo.
- **Error de modelo**: distancia de la predicción nominal al rango medido.

Comparamos con la fase 1b. Si existe un `registro_validacion.csv` de la fase 1b, lo usamos. Si no, tomamos los valores de aquella ejecución, copiados en la celda.

In [ ]:
OBJ = {"retención 1 h [%]": (37, 90), "retención 2 h [%]": (30, 60), "retención 4 h [%]": (0, 10)}
m_nom = metricas_ret(t_nom, ret_nom)
t_f, ret_f, _, _ = simular_comida(PP, masa_c, V0_c, 255, k=12, rtol=1e-11)
m_fino = metricas_ret(t_f, ret_f)
disc = lambda x, lo, hi: 0.0 if lo <= x <= hi else min(abs(x - lo), abs(x - hi))
REG4 = []
for met, (lo, hi) in OBJ.items():
    p5, p95 = np.percentile(MCV[met], [5, 95])
    REG4.append({"versión": "4 · partículas + criba", "comida": "consenso", "métrica": met, "nominal": m_nom[met],
                 "p5": p5, "p95": p95, "objetivo_min": lo, "objetivo_max": hi,
                 "error_numérico": abs(m_nom[met] - m_fino[met]), "error_paramétrico": (p95 - p5) / 2,
                 "error_de_modelo": disc(m_nom[met], lo, hi), "banda_solapa_dato": bool(p95 >= lo and p5 <= hi)})
REG4 = pd.DataFrame(REG4)

if os.path.exists("registro_validacion.csv"):
    previo = pd.read_csv("registro_validacion.csv")
    REG1B = previo[(previo["versión"] == "1b · tampón en equilibrio") & (previo["comida"] == "consenso")]
else:
    REG1B = pd.DataFrame([  # valores de la ejecución de la fase 1b (nominal, p5, p95, error de modelo)
        {"versión": "1b · tampón en equilibrio", "comida": "consenso", "métrica": "retención 1 h [%]", "nominal": 32.08, "p5": 6.28, "p95": 49.22, "objetivo_min": 37, "objetivo_max": 90, "error_de_modelo": 4.92},
        {"versión": "1b · tampón en equilibrio", "comida": "consenso", "métrica": "retención 2 h [%]", "nominal": 1.90, "p5": 0.47, "p95": 18.78, "objetivo_min": 30, "objetivo_max": 60, "error_de_modelo": 28.10},
        {"versión": "1b · tampón en equilibrio", "comida": "consenso", "métrica": "retención 4 h [%]", "nominal": 0.28, "p5": 0.04, "p95": 1.06, "objetivo_min": 0, "objetivo_max": 10, "error_de_modelo": 0.0}])
comp = pd.concat([REG1B, REG4], ignore_index=True)[["versión", "métrica", "nominal", "p5", "p95", "objetivo_min", "objetivo_max", "error_de_modelo"]]
display(comp.style.format(precision=2).hide(axis="index"))
anchos = np.array([hi - lo for lo, hi in OBJ.values()])
e1b = float(np.mean(REG1B["error_de_modelo"].to_numpy() / anchos)); e4 = float(np.mean(REG4["error_de_modelo"].to_numpy() / anchos))
print(f"Error de modelo normalizado medio (retenciones): fase 1b = {e1b:.3f}   ->   fase 4 = {e4:.3f}")
print(f"Error numérico máximo de las retenciones: {REG4['error_numérico'].max():.2e} puntos %")
VERIFICACION.append({"test": "T6 error numérico de las retenciones", "valor": float(REG4["error_numérico"].max()),
                     "criterio": "< 1 punto %", "pasa": float(REG4["error_numérico"].max()) < 1})

# Añadir al registro de validación (sustituye las filas de esta versión si ya existían)
reg = REG4.assign(fase="4", fecha=pd.Timestamp.now().strftime("%Y-%m-%d"))
if os.path.exists("registro_validacion.csv"):
    previo = pd.read_csv("registro_validacion.csv")
    reg = pd.concat([previo[previo["versión"] != "4 · partículas + criba"], reg], ignore_index=True)
reg.to_csv("registro_validacion.csv", index=False)
print(f"registro_validacion.csv: {len(reg)} filas")

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(13, 3.8))
for a, (met, (lo, hi)) in zip(axs, OBJ.items()):
    a.axhspan(lo, hi, color=GRIS_MEDIO, alpha=0.25, lw=0)
    for j, (fila, col, lab) in enumerate([(REG1B[REG1B["métrica"] == met].iloc[0], C2, "1b"),
                                         (REG4[REG4["métrica"] == met].iloc[0], C1, "4")]):
        a.errorbar(j, fila["nominal"], yerr=[[fila["nominal"] - fila["p5"]], [fila["p95"] - fila["nominal"]]],
                   fmt="o", ms=7, capsize=4, color=col)
    a.set_xticks([0, 1]); a.set_xticklabels(["1b: líquido", "4: partículas"]); a.set_xlim(-0.6, 1.6)
    a.set_title(met); a.grid(axis="x", visible=False)
axs[0].set_ylabel("retención [%]")
fig.suptitle("¿Reduce el error de modelo acoplar partículas? (banda gris: rango medido)", fontweight="bold")
fig.tight_layout(); plt.show()

**Lectura.**
- A las 2 h, la predicción nominal queda algo por debajo del rango normal, pero la banda paramétrica lo cubre. A 1 h y a 4 h cae dentro.
- La **criba de partículas** cambia la forma de la curva. Aparece una **fase de latencia**: al principio sale poco, porque casi todo es mayor que el corte del píloro. Después, el vaciado se acelera según la trituración produce partículas pequeñas. Es la forma característica del vaciado de sólidos en gammagrafía, que el modelo de líquido de la fase 1b no podía producir.
- El **error de modelo** de las retenciones baja mucho respecto a la fase 1b (cifras en la tabla). Es la respuesta cuantitativa a tu pregunta inicial: acoplar el modelo de partículas sí reduce el error, y lo hace donde la validación de la 1b señalaba el fallo estructural.
- El **error paramétrico** es grande: la banda es comparable a los rangos normales. Los parámetros mecánicos (resistencia, probabilidad de atrapar, hueco) no están medidos. Para estrecharla harían falta, por ejemplo, ensayos de compresión de la comida digerida a distintos tiempos.
- **El ácido acelera la trituración** (línea punteada). Sin debilitamiento, la rotura se frena a partir de la primera hora y a las 2 h queda más comida (valores impresos). Es el acoplamiento químico–mecánico funcionando en el modelo. Un detalle honesto: la versión *sin* debilitamiento se acerca más al dato de 2 h. Eso no demuestra que el ácido no ablande la comida. Indica que la intensidad del ablandamiento ($X_w$, $w_\infty$) no está acotada y es un candidato claro a medir.
- **Dos limitaciones visibles.**
  - Tras $T_e$, el líquido solo sale al ritmo basal, así que lo que queda (partículas pequeñas y material disuelto) se vacía muy despacio y la curva se aplana.
  - En el mapa de tamaños queda una franja de partículas de más de 6 mm, poco más del 1 % de la masa, que las fuerzas medidas (≤ 2 N) no pueden romper. En el estómago real, ambos restos los vacían las contracciones de "limpieza" del complejo motor migratorio, que el modelo aún no incluye.

## 6. Predicción: 50 g de carne frente a 50 g de pan

Para cerrar el círculo con tu pregunta original, aplicamos el mismo modelo a las comidas de la fase 1b. Las diferencias entre alimentos son **supuestos ilustrativos**, que quedan pendientes de medir:
- La carne es más resistente ($\sigma_{c0}$ = 100 kPa) y se erosiona despacio (pepsina).
- El pan es blando ($\sigma_{c0}$ = 10 kPa), se empapa y se erosiona más deprisa (almidón y amilasa).

In [ ]:
ALIMENTOS = {
    "carne": dict(masa=50.0, kcal=105.0, cambios=dict(sigma_c0=1e5, r_e0=5e-3)),
    "pan":   dict(masa=50.0, kcal=133.0, cambios=dict(sigma_c0=1e4, r_e0=2e-2)),
}
fig, a = plt.subplots(figsize=(8, 3.8))
filas = []
for (nombre, A), col in zip(ALIMENTOS.items(), (C2, C1)):
    p_ = replace(PP, **A["cambios"])
    tt, rr, _, _ = simular_comida(p_, A["masa"], 225.0, A["kcal"])
    bandas = []
    for mu in muestras(40, semilla=11):
        mu = {k: v for k, v in mu.items() if k not in ("sigma_c0", "r_e0")}
        bandas.append(simular_comida(aplicar(p_, mu), A["masa"], 225.0, A["kcal"])[1])
    lo_, hi_ = np.percentile(bandas, [5, 95], axis=0)
    a.fill_between(tt, lo_, hi_, color=col, alpha=0.15, lw=0)
    a.plot(tt, rr, color=col, label=f"50 g de {nombre}")
    m = metricas_ret(tt, rr)
    filas.append({"comida": nombre, "T50 sólidos [min]": m["T50 [min]"], "retención 1 h [%]": m["retención 1 h [%]"],
                  "retención 2 h [%]": m["retención 2 h [%]"]})
a.set(xlabel="t [min]", ylabel="sólido + disuelto en el estómago [%]", title="Predicción ilustrativa (banda: 5–95 sin σc0 ni r_e0)")
a.legend(); fig.tight_layout(); plt.show()
display(pd.DataFrame(filas).style.format(precision=1).hide(axis="index"))

**Lectura.** Con partículas, la carne **sí** permanece más tiempo que el pan a 1 y 2 h, al contrario que en la fase 1b, donde la regla energética la vaciaba antes. El T50 sale parecido porque al principio domina la salida del líquido y de lo disuelto. La intuición clínica de que la carne "pesa" sale ahora de la física: se tritura peor y no pasa por la criba hasta reducirse a menos de 1–2 mm.

## 7. Resumen de verificación

In [ ]:
ver = pd.DataFrame(VERIFICACION)
ver["resultado"] = np.where(ver["pasa"], "✅ pasa", "❌ FALLA")
display(ver.drop(columns="pasa").style.format({"valor": "{:.3g}"}).hide(axis="index"))
n_ok = int(ver["pasa"].sum())
print(f"\n{n_ok}/{len(ver)} tests superados")
assert n_ok == len(ver), "Hay tests de verificación que no pasan"

## 8. Siguientes pasos

**Establecido en esta fase**
- El balance de poblaciones y el Monte Carlo lagrangiano están verificados por separado (solución exacta de rotura, erosión de orden 1, conservación de masa, error estadístico $N^{-1/2}$) y coinciden entre sí.
- El arrastre con Schiller–Naumann y la sedimentación son exactos donde hay solución analítica.
- El flujo LBM de la fase 3 da la frecuencia de visitas a la zona de molienda. Las tensiones del fluido son 5–6 órdenes de magnitud menores que las de contacto: **la trituración es mecánica de pared**.
- La criba de partículas reduce el error de modelo de la validación de la fase 1b.

**Fase 5: modelo integrado.** Juntar en un único bucle de tiempo el flujo LBM (fase 3), el transporte químico (fase 2) y las partículas lagrangianas (esta fase), con el píloro abierto. El pH local de cada partícula vendría del campo de transporte, en lugar de la curva media de la fase 1b. Para la validación, el registro ya tiene las filas de las fases 1b y 4 con las que comparar.